In [1]:
%%capture

import warnings
warnings.filterwarnings('ignore')

import altair as alt
import gcsfs
import pandas as pd

from calitp_portfolio import magics

import prep_data_utils

GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

alt.data_transformers.enable("vegafusion")

In [2]:
#TODO add parameters cell and filter
rtpa = "Sacramento Area Council of Governments"

In [3]:
%%capture_parameters
rtpa

{"rtpa": "Sacramento Area Council of Governments"}


# {rtpa}
## New Transit Performance Metrics

The UCLA Institute of Transportation Studies (UCLA ITS) suggests that:
>Updating the policy and legislation that governs state transit funding could help make expenditures more effective and better aligned with the state’s goals of VMT and GHG reduction, which transit can achieve only through increased ridership.

The UCLA ITS recommends using cost-efficiency metrics (operating expense per VRM/VRH/UPT) and service-effectiveness metrics (passenters per VRM/VRH) to compare transit-oriented vs. auto-oriented markets. 

The charts below display these metrics by different categories.

## Performance Metrics Explained

| Metric type          | Metric example                  | Implicit Goal(s)                       | Advantages                                   | Limitations                                  |
|----------------------|---------------------------------|---------------------------------------|----------------------------------------------|----------------------------------------------|
| Cost-efficiency     | Operating cost per revenue hour (opex_per_vrh) | Reduce costs                         | Useful in both financial and service planning | Favors high labor productivity in dense, congested areas; does not track use |
|                      | Operating cost per revenue mile (opex_per_vrm) |                                       |                                              |                                              |
|                      | Operating cost per vehicle trip (opex_per_upt) |                                       |                                              |                                              |
| Service-effectiveness| Passengers per revenue-vehicle hour (upt_per_vrh) | Increase ridership; reduce poorly patronized service | Useful for service planning; emphasizes what matters to riders | Favors high ridership; does not track costs   |
|                      | Passengers per revenue-vehicle mile (upt_per_vrm) | Increase ridership; reduce low-ridership route miles/segments | Useful for service planning                | Favors high ridership and fast vehicle speeds; does not track costs |


In [4]:
df = pd.read_parquet(
    f"{GCS_FILE_PATH}annual.parquet",
    # should only certain columns be read in? now this table is much larger
    filesystem=gcsfs.GCSFileSystem(),
    columns = [
        "ntd_id", "source_agency", "agency_status", "source_city", 
        "year",
        "mode", "mode_full_name", "type_of_service", "type_of_service_full_name",
        "reporter_type", "reporting_module", "source_state", "primary_uza_name",
        "unlinked_passenger_trips", "vehicle_revenue_hours", "vehicle_revenue_miles",
        "operating_expenses_total",
        "opex_per_vrh", "opex_per_vrm", "opex_per_upt", 
        "upt_per_vrh", "upt_per_vrm",
        "farebox_recovery_ratio", "fare_revenue",
    ]
).pipe(
    prep_data_utils.merge_with_crosswalk
).query(
    f'rtpa_name == "{rtpa}"'
).dropna(
    subset="unlinked_passenger_trips"
)

In [5]:
df.rtpa_name.value_counts()

rtpa_name
Sacramento Area Council of Governments    125
Name: count, dtype: int64

In [6]:
cost_efficiency = ["opex_per_vrh", "opex_per_vrm", "opex_per_upt"]
service_effectiveness = ["upt_per_vrh", "upt_per_vrm"]

## Agency

In [7]:
from great_tables import GT
import gt_extras as gte
import polars as pl

def make_wide_for_nanoplot(
    df: pd.DataFrame, 
    group_cols: list, 
    value_cols: list = cost_efficiency + service_effectiveness
) -> pl.DataFrame:
    df2 = (
        df
        .sort_values(group_cols + ["year"])
        .groupby(group_cols)
        .agg({
            
            c: lambda x: list(x) for c in ["year"] + value_cols
        })
        .reset_index()
    )
    
    df_pl = pl.from_pandas(df2)

    return df_pl

In [8]:
agency_df = prep_data_utils.calculate_efficiency_metrics_by_group(
    df, 
    ["ntd_id", "source_agency", "year", "rtpa_name"]
)
    
agency_pl = make_wide_for_nanoplot(
    agency_df, group_cols = ["ntd_id", "source_agency"]
)

In [9]:
from great_tables import nanoplot_options

cost_nano_options=nanoplot_options(
    data_line_stroke_color="purple",
    data_area_fill_color="white", #lightsteelblue?
    data_point_fill_color="coral",
    data_point_stroke_color="white",
)

service_nano_options=nanoplot_options(
    data_line_stroke_color="steelblue", #default
    data_area_fill_color="white", #lightsteelblue?
    data_point_fill_color="darkorange",
    data_point_stroke_color="white",
)

In [10]:
(
    GT(agency_pl)
    .cols_hide(["year"])
    .fmt_nanoplot(
        columns="opex_per_vrh", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="opex_per_vrm", plot_type="line", missing_vals="gap", options=cost_nano_options
    ).fmt_nanoplot(
        columns="opex_per_upt", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrh", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrm", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).tab_spanner(
        label="cost-efficiency",
        columns=cost_efficiency
    ).tab_spanner(
        label="service-effectiveness",
        columns=service_effectiveness
    ).cols_label(
        ntd_id = "NTD ID",
        source_agency = "Agency",
        opex_per_vrh = "Operating Cost per VRH",
        opex_per_vrm = "Operating Cost per VRM",
        opex_per_upt = "Operating Cost per UPT",
        upt_per_vrh = "Passenger Trips per VRH",
        upt_per_vrm = "Passenger Trips per VRM",
    ).tab_options(table_font_size="14px")
)

GT(_tbl_data=shape: (10, 8)
┌────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───────────┐
│ ntd_id ┆ source_age ┆ year       ┆ opex_per_v ┆ opex_per_v ┆ opex_per_u ┆ upt_per_vr ┆ upt_per_v │
│ ---    ┆ ncy        ┆ ---        ┆ rh         ┆ rm         ┆ pt         ┆ h          ┆ rm        │
│ str    ┆ ---        ┆ list[i64]  ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│        ┆ str        ┆            ┆ list[f64]  ┆ list[f64]  ┆ list[f64]  ┆ list[f64]  ┆ list[f64] │
╞════════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪═══════════╡
│ 90019  ┆ Sacramento ┆ [2018,     ┆ [188.98,   ┆ [14.26,    ┆ [7.31,     ┆ [25.86,    ┆ [1.95,    │
│        ┆ Regional   ┆ 2019, …    ┆ 202.34, …  ┆ 15.17, …   ┆ 8.34, …    ┆ 24.25, …   ┆ 1.82, …   │
│        ┆ Transit    ┆ 2024]      ┆ 239.59]    ┆ 17.24]     ┆ 15.53]     ┆ 15.43]     ┆ 1.11]     │
│        ┆ Di…        ┆            ┆            ┆            ┆            ┆            ┆           │
│ 90061  ┆ Yuba-Sutte ┆ [2018,     ┆ [86.94,    ┆ [5.93,     ┆ [7.1,      ┆ [12.24,    ┆ [0.83,    │
│        ┆ r Transit  ┆ 2019, …    ┆ 88.33, …   ┆ 6.04, …    ┆ 7.83, …    ┆ 11.27, …   ┆ 0.77, …   │
│        ┆ Authority  ┆ 2024]      ┆ 125.92]    ┆ 8.85]      ┆ 14.92]     ┆ 8.44]      ┆ 0.59]     │
│        ┆ …          ┆            ┆            ┆            ┆            ┆            ┆           │
│ 90090  ┆ Yolo       ┆ [2018,     ┆ [113.17,   ┆ [5.85,     ┆ [10.35,    ┆ [10.94,    ┆ [0.57,    │
│        ┆ County Tra ┆ 2019, …    ┆ 115.29, …  ┆ 5.95, …    ┆ 11.17, …   ┆ 10.32, …   ┆ 0.53, …   │
│        ┆ nsportatio ┆ 2024]      ┆ 160.17]    ┆ 7.91]      ┆ 27.79]     ┆ 5.76]      ┆ 0.28]     │
│        ┆ n Dis…     ┆            ┆            ┆            ┆            ┆            ┆           │
│ 90142  ┆ University ┆ [2018,     ┆ [73.42,    ┆ [6.96,     ┆ [1.45,     ┆ [50.75,    ┆ [4.81,    │
│        ┆ of Califor ┆ 2019, …    ┆ 74.71, …   ┆ 7.07, …    ┆ 1.51, …    ┆ 49.51, …   ┆ 4.68, …   │
│        ┆ nia, Davi… ┆ 2024]      ┆ 124.72]    ┆ 12.03]     ┆ 2.64]      ┆ 47.25]     ┆ 4.56]     │
│ 90167  ┆ City of    ┆ [2018,     ┆ [127.97,   ┆ [9.86,     ┆ [38.48,    ┆ [3.33,     ┆ [0.26,    │
│        ┆ Davis      ┆ 2019, …    ┆ 114.22, …  ┆ 9.52, …    ┆ 35.63, …   ┆ 3.21, …    ┆ 0.27, …   │
│        ┆ (DCT) -    ┆ 2024]      ┆ 142.18]    ┆ 13.21]     ┆ 49.55]     ┆ 2.87]      ┆ 0.27]     │
│        ┆ Transit/…  ┆            ┆            ┆            ┆            ┆            ┆           │
│ 90205  ┆ City of    ┆ [2018,     ┆ [135.18,   ┆ [9.26,     ┆ [12.13,    ┆ [11.15,    ┆ [0.76,    │
│        ┆ Elk Grove  ┆ 2019, …    ┆ 136.9, …   ┆ 9.6, …     ┆ 12.75, …   ┆ 10.74, …   ┆ 0.75, …   │
│        ┆ (etran)    ┆ 2021]      ┆ 165.36]    ┆ 11.66]     ┆ 73.64]     ┆ 2.25]      ┆ 0.16]     │
│ 90216  ┆ County of  ┆ [2018,     ┆ [89.38,    ┆ [4.14,     ┆ [18.37,    ┆ [4.87,     ┆ [0.23,    │
│        ┆ Sacramento ┆ 2019, …    ┆ 89.58, …   ┆ 4.49, …    ┆ 19.0, …    ┆ 4.72, …    ┆ 0.24, …   │
│        ┆ Municipal… ┆ 2024]      ┆ 135.4]     ┆ 5.84]      ┆ 49.88]     ┆ 2.71]      ┆ 0.12]     │
│ 90220  ┆ City of    ┆ [2018,     ┆ [166.92,   ┆ [11.37,    ┆ [23.16,    ┆ [7.21,     ┆ [0.49,    │
│        ┆ Folsom     ┆ 2019]      ┆ 163.77]    ┆ 10.94]     ┆ 22.88]     ┆ 7.16]      ┆ 0.48]     │
│        ┆ (FSL)      ┆            ┆            ┆            ┆            ┆            ┆           │
│ 90223  ┆ Paratransi ┆ [2018,     ┆ [81.69,    ┆ [5.51,     ┆ [47.18,    ┆ [1.73,     ┆ [0.12,    │
│        ┆ t, Inc.    ┆ 2019, …    ┆ 89.79, …   ┆ 5.97, …    ┆ 51.0, …    ┆ 1.76, …    ┆ 0.12, …   │
│        ┆            ┆ 2024]      ┆ 101.0]     ┆ 6.45]      ┆ 56.44]     ┆ 1.79]      ┆ 0.11]     │
│ 90314  ┆ Attentive  ┆ [2024]     ┆ [18.82]    ┆ [0.69]     ┆ [9.18]     ┆ [2.05]     ┆ [0.08]    │
│        ┆ Transporta ┆            ┆            ┆            ┆            ┆            ┆           │
│        ┆ tion LLC   ┆          

## Mode

In [11]:
mode_df = prep_data_utils.calculate_efficiency_metrics_by_group(
    df, 
    ["mode", "mode_full_name", "year", "rtpa_name"]
)

mode_pl = make_wide_for_nanoplot(
    mode_df, group_cols = ["mode_full_name"]
)

In [12]:
(
    GT(mode_pl)
    .cols_hide(["year"])
    .fmt_nanoplot(
        columns="opex_per_vrh", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="opex_per_vrm", plot_type="line", missing_vals="gap", options=cost_nano_options
    ).fmt_nanoplot(
        columns="opex_per_upt", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrh", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrm", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).tab_spanner(
        label="cost-efficiency",
        columns=cost_efficiency
    ).tab_spanner(
        label="service-effectiveness",
        columns=service_effectiveness
    ).cols_label(
        mode_full_name = "Mode",
        opex_per_vrh = "Operating Cost per VRH",
        opex_per_vrm = "Operating Cost per VRM",
        opex_per_upt = "Operating Cost per UPT",
        upt_per_vrh = "Passenger Trips per VRH",
        upt_per_vrm = "Passenger Trips per VRM",
    ).tab_options(table_font_size="14px")
)

GT(_tbl_data=shape: (4, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ mode_full_na ┆ year        ┆ opex_per_vr ┆ opex_per_vr ┆ opex_per_up ┆ upt_per_vrh ┆ upt_per_vrm │
│ me           ┆ ---         ┆ h           ┆ m           ┆ t           ┆ ---         ┆ ---         │
│ ---          ┆ list[i64]   ┆ ---         ┆ ---         ┆ ---         ┆ list[f64]   ┆ list[f64]   │
│ str          ┆             ┆ list[f64]   ┆ list[f64]   ┆ list[f64]   ┆             ┆             │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Commuter Bus ┆ [2018,      ┆ [131.05,    ┆ [5.17,      ┆ [8.49,      ┆ [15.43,     ┆ [0.61,      │
│              ┆ 2019, …     ┆ 138.81, …   ┆ 5.68, …     ┆ 9.07, …     ┆ 15.3, …     ┆ 0.63, …     │
│              ┆ 2024]       ┆ 106.59]     ┆ 3.2]        ┆ 23.74]      ┆ 4.49]       ┆ 0.13]       │
│ Demand       ┆ [2018,      ┆ [87.89,     ┆ [6.06, 6.5, ┆ [45.41,     ┆ [1.94,      ┆ [0.13,      │
│ Response     ┆ 2019, …     ┆ 95.21, …    ┆ … 9.83]     ┆ 46.37, …    ┆ 2.05, …     ┆ 0.14, …     │
│              ┆ 2024]       ┆ 152.71]     ┆             ┆ 69.49]      ┆ 2.2]        ┆ 0.14]       │
│ Light Rail   ┆ [2018,      ┆ [285.0,     ┆ [16.04,     ┆ [6.83,      ┆ [41.72,     ┆ [2.35, 2.3, │
│              ┆ 2019, …     ┆ 313.93, …   ┆ 17.58, …    ┆ 7.65, …     ┆ 41.03, …    ┆ … 1.88]     │
│              ┆ 2024]       ┆ 440.53]     ┆ 24.53]      ┆ 13.04]      ┆ 33.79]      ┆             │
│ Motor Bus    ┆ [2018,      ┆ [131.06,    ┆ [10.55,     ┆ [6.38,      ┆ [20.53,     ┆ [1.65,      │
│              ┆ 2019, …     ┆ 138.81, …   ┆ 11.16, …    ┆ 7.15, …     ┆ 19.43, …    ┆ 1.56, …     │
│              ┆ 2024]       ┆ 189.31]     ┆ 14.91]      ┆ 11.4]       ┆ 16.6]       ┆ 1.31]       │
└──────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x7ce957676690>, _boxhead=Boxhead([ColInfo(var='mode_full_name', type=<ColInfoTypeEnum.default: 1>, column_label='Mode', column_align='left', column_width=None), ColInfo(var='year', type=<ColInfoTypeEnum.hidden: 4>, column_label='year', column_align='center', column_width=None), ColInfo(var='opex_per_vrh', type=<ColInfoTypeEnum.default: 1>, column_label='Operating Cost per VRH', column_align='center', column_width=None), ColInfo(var='opex_per_vrm', type=<ColInfoTypeEnum.default: 1>, column_label='Operating Cost per VRM', column_align='center', column_width=None), ColInfo(var='opex_per_upt', type=<ColInfoTypeEnum.default: 1>, column_label='Operating Cost per UPT', column_align='center', column_width=None), ColInfo(var='upt_per_vrh', type=<ColInfoTypeEnum.default: 1>, column_label='Passenger Trips per VRH', column_align='center', column_width=None), ColInfo(var='upt_per_vrm', type=<ColInfoTypeEnum.default: 1>, column_label='Passenger Trips per VRM', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7ce957669410>, _spanners=Spanners([SpannerInfo(spanner_id='cost-efficiency', spanner_level=0, spanner_label='cost-efficiency', spanner_units=None, spanner_pattern=None, vars=['opex_per_vrh', 'opex_per_vrm', 'opex_per_upt'], built=None), SpannerInfo(spanner_id='service-effectiveness', spanner_level=0, spanner_label='service-effectiveness', spanner_units=None, spanner_pattern=None, vars=['upt_per_vrh', 'upt_per_vrm'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7ce963847ed0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7ce9576bff90>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7ce9576bf650>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7ce9576abd90>, <great_tables._gt_data.FormatInfo object at 0x7ce9576d0b90>, <great_tables._gt_data.FormatInfo object at 

## Type of Service

In [13]:
tos_df = prep_data_utils.calculate_efficiency_metrics_by_group(
    df, 
    ["type_of_service", "type_of_service_full_name", "year", "rtpa_name",]
)

tos_pl = make_wide_for_nanoplot(
    tos_df, group_cols = ["type_of_service_full_name"]
)

In [14]:
(
    GT(tos_pl)
    .cols_hide(["year"])
    .fmt_nanoplot(
        columns="opex_per_vrh", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="opex_per_vrm", plot_type="line", missing_vals="gap", options=cost_nano_options
    ).fmt_nanoplot(
        columns="opex_per_upt", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrh", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrm", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).tab_spanner(
        label="cost-efficiency",
        columns=cost_efficiency
    ).tab_spanner(
        label="service-effectiveness",
        columns=service_effectiveness
    ).cols_label(
        type_of_service_full_name = "Type of Service",
        opex_per_vrh = "Operating Cost per VRH",
        opex_per_vrm = "Operating Cost per VRM",
        opex_per_upt = "Operating Cost per UPT",
        upt_per_vrh = "Passenger Trips per VRH",
        upt_per_vrm = "Passenger Trips per VRM",
    ).tab_options(table_font_size="14px")
)

GT(_tbl_data=shape: (4, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ type_of_serv ┆ year        ┆ opex_per_vr ┆ opex_per_vr ┆ opex_per_up ┆ upt_per_vrh ┆ upt_per_vrm │
│ ice_full_nam ┆ ---         ┆ h           ┆ m           ┆ t           ┆ ---         ┆ ---         │
│ e            ┆ list[i64]   ┆ ---         ┆ ---         ┆ ---         ┆ list[f64]   ┆ list[f64]   │
│ ---          ┆             ┆ list[f64]   ┆ list[f64]   ┆ list[f64]   ┆             ┆             │
│ str          ┆             ┆             ┆             ┆             ┆             ┆             │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Directly     ┆ [2018,      ┆ [159.73,    ┆ [12.05,     ┆ [6.95,      ┆ [22.97,     ┆ [1.73,      │
│ Operated     ┆ 2019, …     ┆ 172.92, …   ┆ 12.99, …    ┆ 7.86, …     ┆ 22.01, …    ┆ 1.65, …     │
│              ┆ 2024]       ┆ 231.52]     ┆ 17.15]      ┆ 13.28]      ┆ 17.43]      ┆ 1.29]       │
│ Purchased    ┆ [2018,      ┆ [109.06,    ┆ [6.44,      ┆ [10.67,     ┆ [10.22,     ┆ [0.6, 0.57, │
│ Transportati ┆ 2019, …     ┆ 111.02, …   ┆ 6.61, …     ┆ 11.57, …    ┆ 9.6, … 6.4] ┆ … 0.35]     │
│ on           ┆ 2024]       ┆ 145.93]     ┆ 7.91]       ┆ 22.79]      ┆             ┆             │
│ Purchased    ┆ [2018,      ┆ [117.95,    ┆ [6.02,      ┆ [41.12,     ┆ [2.87,      ┆ [0.15,      │
│ Transportati ┆ 2019, 2020] ┆ 118.97,     ┆ 6.06, 6.43] ┆ 40.66,      ┆ 2.93, 2.94] ┆ 0.15, 0.14] │
│ on - Tax…    ┆             ┆ 137.27]     ┆             ┆ 46.64]      ┆             ┆             │
│ Purchased    ┆ [2022,      ┆ [171.73,    ┆ [8.2, 5.43, ┆ [69.52,     ┆ [2.47,      ┆ [0.12,      │
│ Transportati ┆ 2023, 2024] ┆ 119.71,     ┆ 5.3]        ┆ 44.73,      ┆ 2.68, 2.7]  ┆ 0.12, 0.13] │
│ on - Tra…    ┆             ┆ 113.39]     ┆             ┆ 42.03]      ┆             ┆             │
└──────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x7ce95765d0d0>, _boxhead=Boxhead([ColInfo(var='type_of_service_full_name', type=<ColInfoTypeEnum.default: 1>, column_label='Type of Service', column_align='left', column_width=None), ColInfo(var='year', type=<ColInfoTypeEnum.hidden: 4>, column_label='year', column_align='center', column_width=None), ColInfo(var='opex_per_vrh', type=<ColInfoTypeEnum.default: 1>, column_label='Operating Cost per VRH', column_align='center', column_width=None), ColInfo(var='opex_per_vrm', type=<ColInfoTypeEnum.default: 1>, column_label='Operating Cost per VRM', column_align='center', column_width=None), ColInfo(var='opex_per_upt', type=<ColInfoTypeEnum.default: 1>, column_label='Operating Cost per UPT', column_align='center', column_width=None), ColInfo(var='upt_per_vrh', type=<ColInfoTypeEnum.default: 1>, column_label='Passenger Trips per VRH', column_align='center', column_width=None), ColInfo(var='upt_per_vrm', type=<ColInfoTypeEnum.default: 1>, column_label='Passenger Trips per VRM', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7ce956f10c90>, _spanners=Spanners([SpannerInfo(spanner_id='cost-efficiency', spanner_level=0, spanner_label='cost-efficiency', spanner_units=None, spanner_pattern=None, vars=['opex_per_vrh', 'opex_per_vrm', 'opex_per_upt'], built=None), SpannerInfo(spanner_id='service-effectiveness', spanner_level=0, spanner_label='service-effectiveness', spanner_units=None, spanner_pattern=None, vars=['upt_per_vrh', 'upt_per_vrm'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7ce9576913d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7ce956f13f90>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7ce956f12cd0>, _formats=[<great_tables._gt_data.FormatInfo object at

## Cost-efficiency metrics
Cost-efficiency measures inputs to outputs: For example, the cost of operating an hour of transit service.

Per the UCLA ITS Paper
>Transit-oriented markets (which are predominantly urban), transit service tends to be relatively service-effective. But high operating costs on these (mostly) older, larger systems can inhibit efforts to improve ridership by adding service. In such contexts, assessing systems with an emphasis on **cost-efficiency (i.e., the cost of operating an hour of service)** grounds would provide incentives for agencies to **manage their costs** so as to be able to provide more service with available funding.

### Operating cost per VRH
Lower is better

This section is scatterplot of the raw values, by mode, by type_of_service
* x = raw vehicle revenue hours (log scale)
* y = operating expense total (regular linear scale)
* color = reporter_type
* args in function are a bit confusing with if/else statement (handle once and use for rest of report)
* these are side-by-side charts
* only 1 year is shown, though maybe all the years can be put onto scatterplot?

### Operating cost per VRM
Lower is better

This section is scatterplot of the raw values, by mode, by type_of_service
* x = raw vehicle revenue miles  (log scale)
* y = operating expense total (regular linear scale)
* color = reporter_type

### Operating cost per trip
Lower is better

This section is scatterplot of the raw values, by mode, by type_of_service
* x = Passenger (log scale)
* y = operating expense total (regular linear scale)
* color = reporter_type

## Service-effectiveness metrics
Service-effectiveness measures outputs to consumption: For example, passenger boardings per service hour.

Per the UCLA ITS Paper
>[In] more auto-oriented markets, transit operators tend to be relatively cost-efficient, in that they have lower operating costs but serve fewer riders. In this context, assessing systems with an emphasis on **service-effectiveness (i.e., passenger boardings per service hour)** will motivate operators to **improve ridership** by changing service hours, routes, and fares to better match local demand. Agencies might also implement fare programs with schools and other institutions, and even work with municipalities on improving land use around transit in order to increase the relative attractiveness of transit service.

### Passengers per VRH
Higher is better

This section is scatterplot of the raw values, by mode, by type_of_service
* x = vehicle revenue hours (log scale)
* y = unlinked passenger trips (regular linear scale)
* color = reporter_type

### Passengers per VRM
Higher is better

This section is scatterplot of the raw values, by mode, by type_of_service
* x = vehicle revenue miles (log scale)
* y = unlinked passenger trips (regular linear scale)
* color = reporter_type